In [0]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from xgboost import XGBClassifier

from tqdm import tqdm
import time

from xgboost.callback import TrainingCallback

from sklearn.metrics import confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.metrics import (
    roc_curve,
    roc_auc_score
)
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

In [0]:
df = pd.read_csv("Data/data_processed.csv")

df.head()

In [0]:
X = df.drop(columns=["Class"])
y = df["Class"]

print(X.shape)
print(y.value_counts())

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

In [0]:
scale_pos_weight = (
    y_train.value_counts()[0] /
    y_train.value_counts()[1]
)

scale_pos_weight

In [0]:
model = XGBClassifier(
    objective="binary:logistic",

    eval_metric="logloss",

    n_estimators=300,
    learning_rate=0.05,

    max_depth=6,
    min_child_weight=2,

    subsample=0.8,
    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    random_state=42,

    tree_method="hist"
)

In [0]:
class TQDMCallback(TrainingCallback):

    def __init__(self, n_estimators):
        self.pbar = tqdm(
            total=n_estimators,
            desc="Training XGBoost"
        )

    def after_iteration(
        self,
        model,
        epoch,
        evals_log
    ):
        self.pbar.update(1)
        return False

    def after_training(self, model):
        self.pbar.close()
        return model

In [0]:
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",

    n_estimators=300,
    learning_rate=0.05,

    max_depth=6,
    min_child_weight=2,

    subsample=0.8,
    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    random_state=42,
    tree_method="hist"
)

In [0]:
model.fit(
    X_train,
    y_train
)

In [0]:
y_pred = model.predict(X_test)

In [0]:
acc = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)


print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

In [0]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

In [0]:
y_prob = model.predict_proba(X_test)[:,1]

In [0]:
thresholds = np.arange(
    0.01,
    1.00,
    0.01
)


results = []


for t in thresholds:

    y_pred_t = (y_prob >= t).astype(int)

    acc = accuracy_score(
        y_test,
        y_pred_t
    )

    precision = precision_score(
        y_test,
        y_pred_t,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred_t,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred_t,
        zero_division=0
    )


    results.append([
        t,
        acc,
        precision,
        recall,
        f1
    ])


df_threshold = pd.DataFrame(
    results,
    columns=[
        "Threshold",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ]
)


df_threshold[
    (df_threshold["Precision"] >= 0.8) &
    (df_threshold["Recall"] >= 0.8)
].sort_values(
    by="F1",
    ascending=False
).head(10)

In [0]:
threshold = 0.46

y_pred_final = (
    y_prob >= threshold
).astype(int)

In [0]:
cm = confusion_matrix(
    y_test,
    y_pred_final
)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(
    f"Confusion Matrix (threshold={threshold})"
)

plt.show()

In [0]:
fpr, tpr, thresholds = roc_curve(
    y_test,
    y_prob
)

auc = roc_auc_score(
    y_test,
    y_prob
)


plt.figure(figsize=(7,5))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {auc:.4f}"
)


plt.plot(
    [0,1],
    [0,1],
    linestyle="--"
)


plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve"
)

plt.legend()

plt.show()

In [0]:
precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_prob
)


ap = average_precision_score(
    y_test,
    y_prob
)


plt.figure(figsize=(7,5))

plt.plot(
    recall,
    precision,
    label=f"AP = {ap:.4f}"
)


plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Precision-Recall Curve"
)


plt.legend()

plt.grid()

plt.show()